In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)
print("NIA notebook is working!")

Pandas version: 3.0.5
NIA notebook is working!


In [2]:
import pandas as pd

DATA_PATH = "../data/raw/archive/twcs/twcs.csv"

df = pd.read_csv(DATA_PATH, nrows=10)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (10, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   tweet_id                 10 non-null     int64  
 1   author_id                10 non-null     str    
 2   inbound                  10 non-null     bool   
 3   created_at               10 non-null     str    
 4   text                     10 non-null     str    
 5   response_tweet_id        8 non-null      str    
 6   in_response_to_tweet_id  9 non-null      float64
dtypes: bool(1), float64(1), int64(1), str(4)
memory usage: 622.0 bytes


In [4]:
df["inbound"].value_counts()

inbound
False    5
True     5
Name: count, dtype: int64

In [5]:
df[["inbound", "text"]]

,inbound,text
0,False,@115712 I understand. I would like to assist y...
1,True,@sprintcare and how do you propose we do that
2,True,@sprintcare I have sent several private messag...
3,False,@115712 Please send us a Private Message so th...
4,True,@sprintcare I did.
5,False,@115712 Can you please send us a private messa...
6,True,@sprintcare is the worst customer service
7,False,@115713 This is saddening to hear. Please shoo...
8,True,@sprintcare You gonna magically change your co...
9,False,@115713 We understand your concerns and we'd l...


In [6]:
from collections import Counter

brand_counts = Counter()

for chunk in pd.read_csv(DATA_PATH, usecols=["author_id", "inbound"], chunksize=100_000):
    support_accounts = chunk.loc[chunk["inbound"] == False, "author_id"]
    brand_counts.update(support_accounts)

brand_counts.most_common(20)

[('AmazonHelp', 169840),
 ('AppleSupport', 106860),
 ('Uber_Support', 56270),
 ('SpotifyCares', 43265),
 ('Delta', 42253),
 ('Tesco', 38573),
 ('AmericanAir', 36764),
 ('TMobileHelp', 34317),
 ('comcastcares', 33031),
 ('British_Airways', 29361),
 ('SouthwestAir', 28977),
 ('VirginTrains', 27817),
 ('Ask_Spectrum', 25860),
 ('XboxSupport', 24557),
 ('sprintcare', 22381),
 ('hulu_support', 21872),
 ('sainsburys', 19466),
 ('GWRHelp', 19364),
 ('AskPlayStation', 19098),
 ('ChipotleTweets', 18749)]

In [7]:
spotify_chunks = []

for chunk in pd.read_csv(DATA_PATH, chunksize=100_000):
    spotify_rows = chunk[
        (chunk["author_id"] == "SpotifyCares") |
        (
            (chunk["inbound"] == True) &
            (chunk["text"].str.contains("@SpotifyCares", case=False, na=False))
        )
    ]
    spotify_chunks.append(spotify_rows)

spotify_df = pd.concat(spotify_chunks, ignore_index=True)

print("Spotify rows:", len(spotify_df))
spotify_df.head()

Spotify rows: 74618


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
1,849,115887,True,Tue Oct 31 23:36:20 +0000 2017,@SpotifyCares doesn’t work and i even tried de...,851,848.0
2,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
3,850,115887,True,Tue Oct 31 21:41:37 +0000 2017,@SpotifyCares Premium &amp; when i️ have it on...,848,852.0
4,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0


In [8]:
print("Spotify rows:", len(spotify_df))

Spotify rows: 74618


In [9]:
spotify_df["inbound"].value_counts()

inbound
False    43265
True     31353
Name: count, dtype: int64

In [10]:
spotify_df["created_at"] = pd.to_datetime(spotify_df["created_at"])

print("First tweet:", spotify_df["created_at"].min())
print("Last tweet:", spotify_df["created_at"].max())

C:\Users\ARBAZ SALAM\AppData\Local\Temp\ipykernel_15284\3520734973.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spotify_df["created_at"] = pd.to_datetime(spotify_df["created_at"])


First tweet: 2013-09-18 19:23:17+00:00
Last tweet: 2017-12-03 22:56:04+00:00


In [11]:
spotify_df["year"] = spotify_df["created_at"].dt.year

spotify_df["year"].value_counts().sort_index()

year
2013        3
2015        6
2016       17
2017    74592
Name: count, dtype: int64

In [12]:
customer_tweets = spotify_df[spotify_df["inbound"] == True].copy()

print("Customer tweets:", len(customer_tweets))

customer_tweets[["text"]].head(20)

Customer tweets: 31353


,text
1,@SpotifyCares doesn’t work and i even tried de...
3,@SpotifyCares Premium &amp; when i️ have it on...
5,@SpotifyCares iphone 7+ and i have the most re...
8,"@SpotifyCares Yes, multiple times. No changes...."
10,@SpotifyCares 2/2... and there is no way to ma...
12,@SpotifyCares @115890 Groove Music quits &amp;...
14,@SpotifyCares ok thx
16,is there a way to find non-explicit songs that...
18,@SpotifyCares I tried it on web browser and it...
19,"@SpotifyCares Desktop app still not working, b..."


In [13]:
sample_customers = customer_tweets[["tweet_id", "text"]].sample(
    n=30,
    random_state=42
)

sample_customers

,tweet_id,text
29133,1301944,@SpotifyCares Best response ever! Me @ the use...
30437,1380867,@SpotifyCares Spotify Student + Hulu renews to...
39027,1790160,@SpotifyCares I just dumped the whole thing. I...
40121,1814781,"@SpotifyCares my band's ""related artists"" sect..."
5043,254988,"@SpotifyCares Uh no....that""s not an answer. G..."
27652,1244355,@SpotifyCares Ummm...so it still says Would in...
23021,1049686,@SpotifyCares They don't have an account so it...
24800,1124160,"@SpotifyCares Yep! And individual songs, are f..."
199,9358,@SpotifyCares I'm new at this how do i DELETE ...
21262,967401,@SpotifyCares But I have done the reset functi...


In [14]:
spotify_df[spotify_df["tweet_id"] == 40476][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id


In [15]:
spotify_df[spotify_df["tweet_id"] == 1824347][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40476,1824347,547643,True,@SpotifyCares DM sent,1824346.0


In [16]:
spotify_df[spotify_df["tweet_id"] == 1824346][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40475,1824346,SpotifyCares,False,@547643 Hey Romelio! Can you DM us your accoun...,1824348.0


In [17]:
spotify_df[spotify_df["tweet_id"] == 1824348][
    ["tweet_id", "author_id", "inbound", "text", "in_response_to_tweet_id"]
]

,tweet_id,author_id,inbound,text,in_response_to_tweet_id
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out...,NaN


In [18]:
conversation_ids = [1824348, 1824346, 1824347]

spotify_df[spotify_df["tweet_id"].isin(conversation_ids)][
    ["tweet_id", "author_id", "inbound", "text"]
].sort_values("tweet_id")

,tweet_id,author_id,inbound,text
40475,1824346,SpotifyCares,False,@547643 Hey Romelio! Can you DM us your accoun...
40476,1824347,547643,True,@SpotifyCares DM sent
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out...


In [19]:
pd.set_option("display.max_colwidth", None)

spotify_df[spotify_df["tweet_id"].isin(conversation_ids)][
    ["tweet_id", "author_id", "inbound", "text"]
].sort_values("tweet_id")

,tweet_id,author_id,inbound,text
40475,1824346,SpotifyCares,False,"@547643 Hey Romelio! Can you DM us your account's email address or username, along with your family members'? We'll take a look backstage /RH https://t.co/ldFdZRiNAt"
40476,1824347,547643,True,@SpotifyCares DM sent
40477,1824348,547643,True,@SpotifyCares hello. My family gets kicked out of premium for family because I can't update my address.


## Initial Conversation Insight

A Spotify support case can span multiple tweets and include short
follow-up messages such as "DM sent".

Example:

Customer:
"My family gets kicked out of premium for family because I can't update my address."

SpotifyCares:
"Hey Romelio! Can you DM us your account's email address or username,
along with your family members'? We'll take a look backstage/RH"

Customer:
"DM sent"

This demonstrates that individual tweets should not always be treated
as independent support cases. Conversation context is important for
intent classification, reply generation, and escalation decisions.

## Brand Selection

### Selected Brand: SpotifyCares

I selected SpotifyCares as the target brand for NIA.

The dataset contains 43,265 support tweets from SpotifyCares and
31,353 inbound customer tweets, giving us 74,618 Spotify-related
tweets in total.

SpotifyCares was selected because:

1. It has enough support interactions to build a meaningful
   historical-resolution retrieval system.
2. Customer messages contain recurring support problems that can
   reasonably be grouped into a small intent taxonomy.
3. The conversations include realistic follow-ups, troubleshooting,
   account issues, and short messages such as "DM sent".
4. The scope is manageable for a focused take-home project while
   still providing enough diversity for evaluation.

### Important Dataset Limitation

The SpotifyCares data is heavily concentrated in 2017:

- 2013: 3 tweets
- 2015: 6 tweets
- 2016: 17 tweets
- 2017: 74,592 tweets

Therefore, the historical resolutions used by NIA should not be
assumed to represent Spotify's current support policies. This will
be treated as a limitation when interpreting the final results.

In [21]:
import sys

sys.path.insert(0, "..")

from src.preprocessing import load_spotify_data

spotify_test = load_spotify_data(DATA_PATH)

print("Rows loaded:", len(spotify_test))
print(spotify_test["inbound"].value_counts())

Rows loaded: 74618
inbound
False    43265
True     31353
Name: count, dtype: int64


In [22]:
intent_sample = customer_tweets[
    ["tweet_id", "author_id", "text"]
].sample(
    n=500,
    random_state=42
).reset_index(drop=True)

print("Sample size:", len(intent_sample))
intent_sample.head()

Sample size: 500


,tweet_id,author_id,text
0,1301944,424267,@SpotifyCares Best response ever! Me @ the use of that song https://t.co/TpgOEZyFhe
1,1380867,441421,"@SpotifyCares Spotify Student + Hulu renews tomorrow, but my Hulu is wanting me to pay up to watch anything today. Obviously, renewing the sub tomorrow, so what gives? Why has the Hulu side of things cut me off?"
2,1790160,537736,"@SpotifyCares I just dumped the whole thing. I'd expect its your job to figure this out, isn't it. \nI'd expect some auto-updater left over garbage coupled with audio cache. But who knows"
3,1814781,545167,"@SpotifyCares my band's ""related artists"" section just disappeared :( why tho spotify:artist:0gI3tMeaM9sOpBtXulMbAh"
4,254988,176622,"@SpotifyCares Uh no....that""s not an answer. Give me, and you other customers that have been asking for this for 5 years, the REASON why you don't provide this when every other music streaming service does. Answer?"


In [23]:
labeling_sample = intent_sample.copy()

labeling_sample["intent"] = ""
labeling_sample["notes"] = ""

labeling_sample.to_csv(
    "../evaluation/intent_labeling_sample.csv",
    index=False
)

print("Saved:", len(labeling_sample), "messages")

Saved: 500 messages


In [24]:
from collections import Counter
import re

text = " ".join(customer_tweets["text"].dropna().astype(str)).lower()

words = re.findall(r"\b[a-z]{3,}\b", text)

common_words = Counter(words).most_common(50)

common_words

[('spotifycares', 31621),
 ('the', 15459),
 ('and', 9268),
 ('you', 7174),
 ('for', 6208),
 ('spotify', 6157),
 ('can', 5228),
 ('but', 4691),
 ('have', 4610),
 ('that', 4293),
 ('account', 4255),
 ('this', 4229),
 ('https', 3629),
 ('not', 3620),
 ('with', 3082),
 ('help', 3010),
 ('premium', 3008),
 ('just', 2886),
 ('thanks', 2701),
 ('app', 2578),
 ('when', 2390),
 ('please', 2264),
 ('all', 2170),
 ('get', 2003),
 ('was', 1997),
 ('your', 1936),
 ('why', 1874),
 ('there', 1870),
 ('songs', 1851),
 ('now', 1841),
 ('music', 1771),
 ('from', 1714),
 ('how', 1698),
 ('still', 1692),
 ('what', 1642),
 ('been', 1636),
 ('out', 1547),
 ('thank', 1543),
 ('are', 1540),
 ('song', 1424),
 ('like', 1392),
 ('family', 1386),
 ('any', 1373),
 ('don', 1352),
 ('email', 1320),
 ('will', 1240),
 ('play', 1230),
 ('back', 1214),
 ('hey', 1197),
 ('playlist', 1180)]

In [25]:
from collections import Counter
import re

stop_words = {
    "the", "and", "you", "for", "can", "but", "have", "that",
    "this", "with", "not", "was", "are", "just", "when", "please",
    "all", "get", "got", "has", "had", "from", "your", "they",
    "its", "what", "why", "how", "would", "could", "about",
    "there", "been", "were", "will", "like", "want", "know",
    "thanks", "thank", "help", "spotify", "spotifycares",
    "https", "http", "www"
}

clean_words = []

for message in customer_tweets["text"].dropna().astype(str).str.lower():
    message = re.sub(r"@\w+", " ", message)
    message = re.sub(r"http\S+|www\S+", " ", message)

    words = re.findall(r"\b[a-z]{3,}\b", message)

    clean_words.extend(
        word for word in words
        if word not in stop_words
    )

common_clean_words = Counter(clean_words).most_common(50)

common_clean_words

[('account', 4255),
 ('premium', 3008),
 ('app', 2578),
 ('songs', 1851),
 ('now', 1841),
 ('music', 1771),
 ('still', 1692),
 ('out', 1546),
 ('song', 1424),
 ('family', 1386),
 ('any', 1373),
 ('don', 1352),
 ('email', 1320),
 ('play', 1230),
 ('back', 1214),
 ('hey', 1197),
 ('playlist', 1180),
 ('version', 1174),
 ('one', 1167),
 ('phone', 1126),
 ('need', 1123),
 ('iphone', 1107),
 ('tried', 1069),
 ('only', 1065),
 ('student', 1065),
 ('new', 1017),
 ('same', 989),
 ('work', 979),
 ('guys', 945),
 ('time', 923),
 ('again', 909),
 ('using', 895),
 ('doesn', 884),
 ('album', 874),
 ('update', 850),
 ('issue', 835),
 ('ios', 834),
 ('charged', 812),
 ('use', 809),
 ('problem', 778),
 ('trying', 770),
 ('try', 767),
 ('working', 758),
 ('playlists', 749),
 ('says', 746),
 ('having', 742),
 ('some', 739),
 ('already', 734),
 ('does', 728),
 ('change', 724)]

In [26]:
keywords = [
    "premium",
    "family",
    "student",
    "charged",
    "account",
    "password",
    "app",
    "iphone",
    "play",
    "playlist"
]

for keyword in keywords:
    print("\n" + "=" * 70)
    print("KEYWORD:", keyword.upper())
    print("=" * 70)

    matches = customer_tweets[
        customer_tweets["text"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ][["tweet_id", "text"]].sample(
        n=min(5, len(customer_tweets[
            customer_tweets["text"].str.contains(
                keyword,
                case=False,
                na=False
            )
        ])),
        random_state=42
    )

    for _, row in matches.iterrows():
        print("-", row["text"])


KEYWORD: PREMIUM
- @SpotifyCares upgraded to premium and it still says I have a free account. Fix?
- @SpotifyCares Hey all, I seem to be having a bit of issue with Spotify premium on my laptop. Won't play anything. Getting diff errors too
- @SpotifyCares I work at Starbucks and get premium free. I recently downloaded the Spotify desktop app, logged in there, and now it's gone.
- @SpotifyCares English then: can I still get student discount when I already have premium?
- @SpotifyCares may I just also ask if the 9 peso premium thing is only applicable to card?

KEYWORD: FAMILY
- @SpotifyCares I was invited in family but I can't upgrade my account WHY?
- @SpotifyCares I can't add family to my newly upgraded premium account, when I click the link to edit family members, it loads a blank page!
- @SpotifyCares 
Ho un account family. Quello dei due miei familiari sono passati a Free. Cosa devo fare ?
- @SpotifyCares I just invited someone to Premium for Family, but the provided code doesn't w

## NIA Intent Taxonomy — v1

Based on exploratory analysis of SpotifyCares customer conversations,
I define the following 10 intents for the first version of NIA.

| Intent | Description |
|---|---|
| `premium_subscription` | Problems or questions about Premium subscriptions, upgrades, downgrades, or Premium status. |
| `family_plan` | Spotify Family plan invitations, members, address/eligibility, or Family subscription problems. |
| `student_plan` | Student discount, student verification, eligibility, or Student Premium issues. |
| `billing_payment` | Unexpected charges, payment failures, refunds, incorrect pricing, trials, or payment-method issues. |
| `account_login_security` | Login, password reset, account access, hacked/compromised accounts, email changes, or account recovery. |
| `app_technical_issue` | App crashes, installation/update problems, device compatibility, errors, or technical failures. |
| `playback_music_issue` | Problems playing music, songs, albums, offline playback, or playback interruptions. |
| `playlist_library` | Playlist creation, transfer, sorting, saved music, playlist display, or library-management issues. |
| `feature_information` | Questions or requests about Spotify features, availability, functionality, or product behavior. |
| `feedback_other` | General feedback, praise, complaints, unclear requests, or messages that do not fit the other categories. |

### Labeling rule

Assign the intent based on the customer's **primary problem**.

When multiple issues appear, choose the issue that requires the
most important support action.

If there is insufficient information to determine the customer's
problem, use `feedback_other` rather than guessing.

Short follow-ups such as "DM sent", "thanks", or "I tried that"
should be interpreted using conversation context when available.

In [35]:
from src.conversation import get_conversation

conversation = get_conversation(1824347, spotify_df)
conversation[["tweet_id", "inbound", "text"]]

KeyError: "None of [Index(['tweet_id', 'inbound', 'text'], dtype='str')] are in the [columns]"

In [32]:
print("Rows returned:", len(conversation))
print("Columns returned:", conversation.columns.tolist())
print(conversation)

Rows returned: 0
Columns returned: []
Empty DataFrame
Columns: []
Index: []


In [33]:
print(spotify_df["tweet_id"].dtype)

tweet_map = spotify_df.set_index("tweet_id").to_dict("index")

print(type(next(iter(tweet_map.keys()))))
print(1824347 in tweet_map)
print(spotify_df.index[spotify_df["tweet_id"] == 1824347].tolist())

int64
<class 'int'>
True
[40476]


In [36]:
import inspect
from src.conversation import get_conversation

print(inspect.getsource(get_conversation))

def get_conversation(tweet_id: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Reconstruct the conversation leading up to a given tweet.

    Follows the in_response_to_tweet_id chain backwards and
    returns messages in chronological order.
    """

    tweet_map = df.set_index("tweet_id").to_dict("index")

    conversation = []
    current_id = int(tweet_id)

    while current_id in tweet_map:
        tweet = tweet_map[current_id].copy()
        tweet["tweet_id"] = current_id

        conversation.append(tweet)

        parent_id = tweet.get("in_response_to_tweet_id")

        if pd.isna(parent_id) or parent_id is None:
            break

        current_id = int(float(parent_id))

    conversation.reverse()

    return pd.DataFrame(conversation)



In [37]:
tweet_map = spotify_df.set_index("tweet_id").to_dict("index")

current_id = 1824347

print("Starting ID:", current_id)
print("Found:", current_id in tweet_map)

if current_id in tweet_map:
    tweet = tweet_map[current_id]
    print("Text:", tweet["text"])
    print("Parent ID:", tweet["in_response_to_tweet_id"])

Starting ID: 1824347
Found: True
Text: @SpotifyCares DM sent
Parent ID: 1824346.0


In [38]:
print(1824346 in tweet_map)

if 1824346 in tweet_map:
    parent = tweet_map[1824346]
    print("Parent text:", parent["text"])
    print("Parent ID:", parent["in_response_to_tweet_id"])

True
Parent text: @547643 Hey Romelio! Can you DM us your account's email address or username, along with your family members'? We'll take a look backstage /RH https://t.co/ldFdZRiNAt
Parent ID: 1824348.0


In [39]:
print(1824348 in tweet_map)

if 1824348 in tweet_map:
    original = tweet_map[1824348]
    print("Original text:", original["text"])
    print("Parent ID:", original["in_response_to_tweet_id"])

True
Original text: @SpotifyCares hello. My family gets kicked out of premium for family because I can't update my address.
Parent ID: nan


In [40]:
import importlib
import src.conversation

importlib.reload(src.conversation)

conversation = src.conversation.get_conversation(1824347, spotify_df)

print("Rows:", len(conversation))
print(conversation[["tweet_id", "inbound", "text"]])

Rows: 3
   tweet_id  inbound  \
0   1824348     True   
1   1824346    False   
2   1824347     True   

                                                                                                                                                                    text  
0                                                                @SpotifyCares hello. My family gets kicked out of premium for family because I can't update my address.  
1  @547643 Hey Romelio! Can you DM us your account's email address or username, along with your family members'? We'll take a look backstage /RH https://t.co/ldFdZRiNAt  
2                                                                                                                                                  @SpotifyCares DM sent  
